# PESStore HDF5 IO Bottleneck Test
Benchmarking the I/O performance of `PESStore` to ensure write speeds exceed 100 MB/s when writing a 1GB dummy trajectory dataset.

In [1]:
import sys
import os
import time
from pathlib import Path
import numpy as np

repo_path = r"D:\Gdrive\__CoChem\GitHub-Repo"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
sys.path.insert(0, os.path.join(repo_path, "CoChem-KINETIC"))

from kinetic_core.cochem_pes_store import PESStore

In [2]:
# Generate ~1GB dataset
n_steps = 10000
N_ATOMS = 2200

print("Generating 1GB dummy trajectory dataset in RAM...")
coords = np.random.rand(n_steps, N_ATOMS, 3)
energy = np.random.rand(n_steps)
gradient = np.random.rand(n_steps, N_ATOMS, 3)

size_mb = (coords.nbytes + energy.nbytes + gradient.nbytes) / (1024 * 1024)
print(f"Data size: {size_mb:.2f} MB")

Generating 1GB dummy trajectory dataset in RAM...


Data size: 1007.16 MB


In [3]:
store_path = Path("test_1gb.h5")
if store_path.exists():
    store_path.unlink()

store = PESStore(store_path)

print("Writing to PESStore...")
start_time = time.time()
store.append_batch(coords, energy, gradient_batch=gradient)
end_time = time.time()

write_time = end_time - start_time
speed_mb_s = size_mb / write_time

print(f"Write time: {write_time:.2f} s")
print(f"Write speed: {speed_mb_s:.2f} MB/s")

# Cleanup
if store_path.exists():
    store_path.unlink()

# Validation assertion
assert speed_mb_s > 100, f"Performance bottleneck! Write speed was only {speed_mb_s:.2f} MB/s"
print("PASS: Write speed > 100 MB/s validation successful.")

Writing to PESStore...


Write time: 4.23 s
Write speed: 237.93 MB/s
PASS: Write speed > 100 MB/s validation successful.
